<a href="https://colab.research.google.com/github/venkatasai-eng/MLA0305-REINFORCEMENT-LEARNING-/blob/main/EXP_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers

np.random.seed(42)
tf.random.set_seed(42)

states = 5
actions = 2
episodes = 100
gamma = 0.9

def step(s, a):
    ns = min(s + 1, 4) if a == 1 else max(s - 1, 0)
    r = 10 if ns == 4 else -1
    return ns, r, ns == 4

def encode(s):
    x = np.zeros(states)
    x[s] = 1
    return x

inp = layers.Input(shape=(states,))
x = layers.Dense(16, activation="relu")(inp)
actor = layers.Dense(actions, activation="softmax")(x)
critic = layers.Dense(1)(x)

model = tf.keras.Model(inp, [actor, critic])
opt = tf.keras.optimizers.Adam(0.001)

for ep in range(episodes):
    s = 0

    while True:
        x = encode(s).reshape(1, -1)

        with tf.GradientTape() as tape:
            prob, value = model(x)
            a = np.random.choice(actions, p=prob[0].numpy())

            ns, r, done = step(s, a)
            _, nv = model(encode(ns).reshape(1, -1))

            target = r if done else r + gamma * nv[0, 0]
            advantage = target - value[0, 0]

            actor_loss = -tf.math.log(prob[0, a] + 1e-8) * tf.stop_gradient(advantage)
            critic_loss = tf.square(advantage)

            loss = actor_loss + 0.5 * critic_loss

        grads = tape.gradient(loss, model.trainable_variables)
        opt.apply_gradients(zip(grads, model.trainable_variables))

        s = ns

        if done:
            break

print("A2C Training Completed")

for s in range(states):
    p, v = model(encode(s).reshape(1, -1))
    print("State", s, "Action", np.argmax(p[0]), "Value", round(float(v[0, 0]), 2))

A2C Training Completed
State 0 Action 1 Value -0.34
State 1 Action 1 Value 0.42
State 2 Action 1 Value 1.19
State 3 Action 1 Value 2.82
State 4 Action 1 Value 1.0
